In [5]:
# The goal of the EDA is to understand the schema before building feature engineering for the
# pre-match (Dixon-Coles) and live in-play models. As with any project I have built this such
# that I understand the data before feature engineering as it becomes easier to spot outliers
# now rather than mid-feature engineering.

In [25]:
import pandas as pd
from statsbombpy import sb

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

COMPETITION_ID = 11 # This is La Liga
SEASON_ID = 27 # This is the 2015/2016 season

In [ ]:
# Here I want to find out what the actual match-level data looks like

matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
print(f"Total matches: {len(matches)}")
print(f"Columns: {list(matches.columns)}")
matches.head()

In [ ]:
# Here I want to check the team coverage, as the gaps here would matter a lot for the Dixon-Coles model
team_counts = pd.concat([
    matches["home_team"], matches["away_team"]
]).value_counts()
print(team_counts)

In [ ]:
# For this code I want to find out what do I have to work with
# for the pre-match model's target variable
# we will print the first ten matches in the dataset with key info that i am interested in
print(matches[["match_date", "home_team", "away_team",
               "home_score","away_score"]].head(10))
print("\nDate range:", matches["match_date"].min(), "to", matches["match_date"].max())

In [ ]:
# Specifically look at one match of interest and understand the raw schema
sample_match_id = matches.iloc[0]["match_id"]
print(f"Sample match: {matches.iloc[0]['home_team']} vs. {matches.iloc[0]['away_team']}"
      f" (match_id={sample_match_id})")

events = sb.events(match_id=sample_match_id)
print(f"\nTotal events in this match: {len(events)}")
print(f"Columns ({len(events.columns)}): {list(events.columns)}")

In [ ]:
# What kind of events exist, and how often do they occur?
print(events["type"].value_counts())

In [ ]:
# explore data that will be used to build the xG for the models
shots= events[events["type"]== "Shot"]
print(f"Total shots in this match: {len(shots)}")

In [ ]:
# Inspect shot-specific columns
shot_cols = [c for c in shots.columns if "shot" in c.lower()]
print(f"\nShot-specific columns: {shot_cols}")
shots[["minute", "second", "team", "player"]+ shot_cols].head()

In [ ]:
# find the xG in question and check for sanity
print(shots["shot_statsbomb_xg"].describe())

In [ ]:
# Find possession and timing that is needed for the live in-play feature window
# 'possession' groups events into possession chains; 'minute'/'second' give
#  the timeline we need to build rolling windows for the live model.
print(events[["minute", "second", "possession", "possession_team", 
              "type", "team", "player"]].head(20))

In [ ]:
# Find the cards and substitutions, which are needed as live model features
cards = events[events["type"].isin(["Foul Committed"])]
print("Foul committed sample (check for card info nested in qualifiers):")
print(cards[["minute", "team", "player", "foul_committed_card"]].dropna(
    subset=["foul_committed_card"]
).head() if "foul_committed_card" in cards.columns else "No card column found — check schema")

subs = events[events["type"] == "Substitution"]
print(f"\nSubstitutions in this match: {len(subs)}")

In [ ]:
missing_pct = (events.isna().mean() * 100).sort_values(ascending=False)
print("Top 20 most sparse columns:")
print(missing_pct.head(20))

In [ ]:
# Checck lineups, formations and the starting 11 which is useful for finding context but isn't strong
# in terms of priority
lineups = sb.lineups(match_id=sample_match_id)
print(f"Teams in lineup data: {list(lineups.keys())}")
sample_team = list(lineups.keys())[0]
print(lineups[sample_team].head())